In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


from squlearn import Executor
from squlearn.encoding_circuit import (
    KyriienkoEncodingCircuit,
)
from squlearn.encoding_circuit.layered_encoding_circuit import Layer
from squlearn.observables import SummedPaulis
from squlearn.qnn import QNNRegressor, ODELoss, get_lr_decay
from squlearn.optimizers import Adam
from scipy.integrate import odeint

from squlearn.qnn.lowlevel_qnn import LowLevelQNN


In [4]:
executor = Executor("pennylane")
def FQK_circuit(circuit1, circuit2):
    return circuit1.compose(circuit2.inverse())

from squlearn.observables import CustomObservable
def P0_squlearn(num_qubits):
    """
    Create the P0 observable: (|0><0|)^\otimes n for the quantum circuit in the format of the squlearn library. 
    Note that |0><0| = 0.5*(I + Z) 

    Parameters:
    num_qubits: int, the number of qubits in the quantum circuit.

    return:
    - CustomObservable: The P0 observable in the format of the squlearn library.
    - coefficients: The coefficients of the P0 observable to be used in the QNN squlearn evaluation

    """
    from qiskit.quantum_info import SparsePauliOp
    
    P0_single_qubit = SparsePauliOp.from_list([("Z", 0.5), ("I", 0.5)])
    P0_temp = P0_single_qubit
    for i in range(1, num_qubits):
        P0_temp = P0_temp.expand(P0_single_qubit)
    observable_tuple_list = P0_temp.to_list()
    pauli_str = [observable[0] for observable in observable_tuple_list]    
    return CustomObservable(num_qubits, pauli_str, parameterized=True)
def to_FQK_circuit_format(x, y = None):
    """
    Transforms an input array of shape (n, m) into an array of shape (n*n, 2*m),
    where each row consists of all possible ordered pairs of rows from the input array.

    Parameters:
    x (numpy.ndarray): An input array of shape (n, m), where n is the number of samples
                       and m is the number of features.

    Returns:
    numpy.ndarray: A transformed array of shape (n*n, 2*m), containing all possible
                   ordered pairs of rows from x.

    Example:
    --------
    >>> x = np.array([[1], 
    ...               [2], 
    ...               [3]])
    >>> to_proper_format(x)
    array([[1, 1],
           [2, 1],
           [3, 1],
           [1, 2],
           [2, 2],
           [3, 2],
           [1, 3],
           [2, 3],
           [3, 3]])
    """
    if y is None:
        y = x
        n, m = x.shape
        x_rep = np.repeat(x, n, axis=0)  # Repeat each row n times
        x_tile = np.tile(x, (n, 1))      # Tile the entire array n times
    else:
        n, m = x.shape
        n2, m2 = y.shape
        x_rep = np.repeat(x, n2, axis=0)
        x_tile = np.tile(y, (n, 1))
    result = np.hstack((x_rep, x_tile))
    return result


N_samples = 3
n_dim = 1
n_qubits = 2
x_space = np.random.rand(N_samples, n_dim)
N_samples_y = 3
y_space = np.random.rand(N_samples_y, n_dim)

circuit1 = KyriienkoEncodingCircuit(
    num_qubits=n_qubits,
    encoding_style="chebyshev_tower",
    variational_arrangement="HEA",
    num_features=n_dim,
    num_encoding_layers=1,
    num_variational_layers=2,
)

coef = np.array([1/2**n_qubits for i in range(2**n_qubits)])
low_qnn = LowLevelQNN(FQK_circuit(circuit1, circuit1), P0_squlearn(n_qubits), executor=executor)
params = np.random.rand(low_qnn.num_parameters)
output_f = low_qnn.evaluate(to_FQK_circuit_format(x_space, y_space), params, coef, "f")["f"]

print(output_f.reshape(N_samples, N_samples_y))

from squlearn.kernel import FidelityKernel
fqk = FidelityKernel(circuit1, executor=executor)
print("Normal squlearn", fqk.evaluate(x_space, y_space))

print("LowQNN implementatio", fqk.evaluate_derivatives(x_space, y_space, values=["K"])["K"])



[[0.9959887  0.9147897  0.99946962]
 [0.96913396 0.96674299 0.98152844]
 [0.23434998 0.5519219  0.26542284]]
Normal squlearn [[0.9959887  0.9147897  0.99946962]
 [0.96913396 0.96674299 0.98152844]
 [0.23434998 0.5519219  0.26542284]]
LowQNN implementatio [[0.9959887  0.9147897  0.99946962]
 [0.96913396 0.96674299 0.98152844]
 [0.23434998 0.5519219  0.26542284]]


In [3]:
fqk.evaluate_derivatives(x_space, y_space, values=["dKdp"])

{'x': array([[0.96415455, 0.6831316 ],
        [0.96415455, 0.53889554],
        [0.96415455, 0.69382023],
        [0.8662198 , 0.6831316 ],
        [0.8662198 , 0.53889554],
        [0.8662198 , 0.69382023],
        [0.81091537, 0.6831316 ],
        [0.81091537, 0.53889554],
        [0.81091537, 0.69382023]]),
 'param': array([ 0.61340858,  2.70414933,  1.29136267,  0.56401871, -0.95938209,
         1.8333595 , -0.78430223,  4.92316472,  5.82655809, -1.46471707,
         3.66592495,  0.36310427]),
 'param_op': array([0.25, 0.25, 0.25, 0.25]),
 'dKdp': array([[[-1.82291039e-16, -5.55111512e-17,  1.52655666e-16],
         [ 2.01069387e-17, -5.55111512e-17,  0.00000000e+00],
         [ 4.85722573e-17,  0.00000000e+00,  2.60208521e-17]],
 
        [[-7.26415456e-18,  4.16333634e-17, -3.46944695e-17],
         [-3.03560757e-17, -8.67361738e-18,  1.82145965e-17],
         [-6.77672640e-19,  6.93889390e-18, -6.93889390e-18]],
 
        [[ 9.32413868e-18, -3.46944695e-18,  1.73472348e-18],
  